# Module 9 Demo — Tools and Agents in Generative AI Applications


1. **Function/tool calling with Amazon Bedrock Converse API**
2. **A ReAct-style reason → act → observe loop**
3. **The same idea implemented with Strands Agents**
4. **MCP: connect a Strands agent to the AWS Documentation MCP Server**
5. **A2A: understand where agent-to-agent interoperability fits**
6. **Cleanup**

### Suggested delivery
- Core demo (Sections 1–3): ~6–7 minutes
- MCP demo (Section 4): ~3–4 minutes
- A2A architecture discussion: ~2 minutes if time permits



---

## Source projects used as references

- Strands Agents samples: https://github.com/strands-agents/samples
- Strands Agents Python docs: https://strandsagents.com/docs/user-guide/quickstart/python/
- AWS Labs MCP servers: https://github.com/awslabs/mcp
- AWS Documentation MCP Server: https://github.com/awslabs/mcp/tree/main/src/aws-documentation-mcp-server
- AWS agent interoperability sample: https://github.com/aws-samples/sample-agent-interoperability

The examples below are simplified for classroom delivery. They are not intended as production architecture.

## 0. What students should see

Use this picture throughout the demo:

```text
User
  |
  v
LLM / Agent
  |
  | decides which tool is needed
  v
Tool request
  |
  | application/framework executes tool
  v
Tool result / Observation
  |
  v
LLM / Agent
  |
  +----> another tool if needed
  |
  v
Final response
```

### Key distinction

**Function calling:** the model proposes a function/tool call; your application executes it.

**Agent:** the system can repeat reasoning, tool selection, action, and observation until the goal is complete.

# 1. Environment setup

### Prerequisites

- Python 3.10+
- Use the first configuration cell to select an AWS profile/runtime role, or enter temporary credentials through hidden prompts.
- Bedrock model access and `bedrock:InvokeModel` for the selected model in the selected Region.
- For Section 4 (MCP): `uv` / `uvx` installed and internet access for the AWS Documentation MCP Server.

An AWS administrator should assign the learner execution role before class; learners should not receive IAM role-creation or administrator permissions. See `setup/bedrock_course_roles.md` in the course repository. Strands uses the configured AWS credential chain for Bedrock.


In [ ]:
import sys
import subprocess
print('Kernel Python:', sys.executable)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'boto3', 'strands-agents', 'strands-agents-tools', 'mcp'])
print('Installation complete.')

In [ ]:
import os
import json
import platform
from datetime import datetime, timezone
from decimal import Decimal
import getpass
import boto3

AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
AWS_AUTH_METHOD = "profile"  # Change to "keys" only when a profile is unavailable.

if AWS_AUTH_METHOD == "profile":
    AWS_PROFILE_NAME = os.getenv("AWS_PROFILE") or None
    session = boto3.Session(profile_name=AWS_PROFILE_NAME, region_name=AWS_REGION)
    boto3.setup_default_session(profile_name=AWS_PROFILE_NAME, region_name=AWS_REGION)
    print("Using AWS profile/default credential chain.")
elif AWS_AUTH_METHOD == "keys":
    print("Enter temporary credentials only. Inputs are hidden and are not saved in this notebook.")
    access_key_id = getpass.getpass("AWS access key ID: ")
    secret_access_key = getpass.getpass("AWS secret access key: ")
    session_token = getpass.getpass("AWS session token (leave blank if not applicable): ")
    credential_args = {
        "aws_access_key_id": access_key_id,
        "aws_secret_access_key": secret_access_key,
        "region_name": AWS_REGION,
    }
    if session_token:
        credential_args["aws_session_token"] = session_token
    session = boto3.Session(**credential_args)
    boto3.setup_default_session(**credential_args)
    del access_key_id, secret_access_key, session_token, credential_args
    print("Using temporary credentials for this kernel session.")
else:
    raise ValueError('AWS_AUTH_METHOD must be "profile" or "keys".')

MODEL_ID = os.getenv("MODEL_ID", "global.anthropic.claude-sonnet-4-6")

print("AWS region :", AWS_REGION)
print("Model ID   :", MODEL_ID)
print("Python OS  :", platform.system())




Verify credentials and Bedrock access before class. This avoids spending class time on IAM/model-access troubleshooting.

In [ ]:
credentials = session.get_credentials()
print("AWS credentials detected:", credentials is not None)

bedrock = session.client("bedrock-runtime")

# 2. Demo 1 — Function calling with Amazon Bedrock Converse

We will deliberately implement the tool loop ourselves first.

This makes one concept unmistakable:

> **The model does not execute the Python function. Our client application executes it.**

We provide two tools:
- `calculator`
- `get_ebike_price`

The e-bike catalog is local on purpose. It keeps the demo deterministic and lets students focus on **tool calling**, not external APIs.

In [ ]:
EBIKE_CATALOG = {
    "CityLite": {"price": 899.0, "range_km": 55},
    "TrailPro": {"price": 1499.0, "range_km": 75},
    "CargoMax": {"price": 1899.0, "range_km": 65},
}

def calculator(operation: str, a: float, b: float) -> float:
    """Perform a basic arithmetic operation."""
    if operation == "add":
        return a + b
    if operation == "subtract":
        return a - b
    if operation == "multiply":
        return a * b
    if operation == "divide":
        return a / b
    raise ValueError(f"Unsupported operation: {operation}")

def get_ebike_price(model_name: str) -> dict:
    """Return product details for an e-bike model."""
    if model_name not in EBIKE_CATALOG:
        return {"found": False, "available_models": list(EBIKE_CATALOG)}
    return {"found": True, "model": model_name, **EBIKE_CATALOG[model_name]}

TOOL_HANDLERS = {
    "calculator": calculator,
    "get_ebike_price": get_ebike_price,
}

tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "calculator",
                "description": "Perform arithmetic. Use this when a numeric calculation is required.",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "operation": {
                                "type": "string",
                                "enum": ["add", "subtract", "multiply", "divide"],
                            },
                            "a": {"type": "number"},
                            "b": {"type": "number"},
                        },
                        "required": ["operation", "a", "b"],
                    }
                },
            }
        },
        {
            "toolSpec": {
                "name": "get_ebike_price",
                "description": "Get price and range information for an e-bike model.",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "model_name": {
                                "type": "string",
                                "description": "Exact e-bike model name.",
                            }
                        },
                        "required": ["model_name"],
                    }
                },
            }
        },
    ]
}

print("Tools exposed to the model:")
for t in tool_config["tools"]:
    print("-", t["toolSpec"]["name"])

### First prompt

Ask something that requires **both retrieval and calculation**:

> “What would 3 TrailPro bikes cost before tax?”

Expected behavior:

```text
LLM
 ↓
get_ebike_price("TrailPro")
 ↓
1499
 ↓
calculator("multiply", 1499, 3)
 ↓
4497
 ↓
final answer
```

Depending on the model, it may issue the calls sequentially or use available context efficiently. The teaching point remains the same: **our code executes the tools**.

In [ ]:
def execute_tool_call(tool_use):
    tool_name = tool_use["name"]
    tool_input = tool_use["input"]
    tool_use_id = tool_use["toolUseId"]

    print(f"\nCLIENT EXECUTES TOOL: {tool_name}")
    print("Input:", tool_input)

    try:
        result = TOOL_HANDLERS[tool_name](**tool_input)
        status = "success"
    except Exception as exc:
        result = {"error": str(exc)}
        status = "error"

    print("Observation:", result)

    return {
        "toolResult": {
            "toolUseId": tool_use_id,
            "content": [{"json": result if isinstance(result, dict) else {"result": result}}],
            "status": status,
        }
    }


def run_bedrock_tool_loop(user_prompt, max_steps=8):
    messages = [{"role": "user", "content": [{"text": user_prompt}]}]

    for step in range(1, max_steps + 1):
        print(f"\n{'='*18} STEP {step} {'='*18}")

        response = bedrock.converse(
            modelId=MODEL_ID,
            messages=messages,
            toolConfig=tool_config,
            inferenceConfig={"temperature": 0.1, "maxTokens": 800},
        )

        assistant_message = response["output"]["message"]
        messages.append(assistant_message)

        tool_uses = [
            block["toolUse"]
            for block in assistant_message["content"]
            if "toolUse" in block
        ]

        text_blocks = [
            block["text"]
            for block in assistant_message["content"]
            if "text" in block
        ]

        if text_blocks:
            print("MODEL:", "\n".join(text_blocks))

        if not tool_uses:
            print("\nFINAL RESPONSE:")
            print("\n".join(text_blocks))
            return response, messages

        tool_results = [execute_tool_call(tool_use) for tool_use in tool_uses]
        messages.append({"role": "user", "content": tool_results})

    raise RuntimeError("Maximum tool-loop steps reached")

In [ ]:
response, conversation = run_bedrock_tool_loop(
    "What would 3 TrailPro e-bikes cost before tax? "
    "Use tools for product data and arithmetic."
)




1. **Who chose `get_ebike_price`?** → the model.
2. **Who actually executed `get_ebike_price()`?** → our Python application.
3. **What did the model receive after execution?** → the tool result / observation.
4. **Why might it call another tool?** → the goal is not complete yet.

This is the bridge from **function calling** to the **agentic loop**.

# 3. Demo 2 — Make the ReAct-style loop visible

ReAct combines the ideas of **reasoning and acting**.

We should not try to expose hidden chain-of-thought. Instead, teach the observable control loop:

```text
Goal
 ↓
Choose next action
 ↓
Use tool
 ↓
Observe tool result
 ↓
Enough information?
 ├─ No → choose another action
 └─ Yes → answer
```

The function from Demo 1 already implements this loop. Now give it a goal that naturally requires several steps.

# 4. Demo 3 — Replace our orchestration code with Strands Agents

In the previous demos, **we wrote the orchestration loop**.

Now use Strands Agents and keep the same mental model:

```text
Prompt + Model + Tools
          |
          v
     Strands Agent
          |
          v
   model-driven loop
```

Strands handles model invocation, tool schemas, tool execution, observations, and repeated calls until the task is complete.

In [ ]:
prompt = """
I am buying 2 CityLite bikes and 1 CargoMax bike.

Use the product tool to get the prices.
Then use the calculator tool for every arithmetic operation.
Tell me the total price before tax and which model has the longer range.
"""

response, conversation = run_bedrock_tool_loop(prompt)



Point at the output and narrate only what is observable:

- The model selected a tool.
- The application executed it.
- The tool returned an observation.
- The model received the observation.
- The model decided whether another action was required.
- Eventually it produced the final answer.

>  “Function calling is one capability. The repeated decision/action/observation cycle is what starts to feel agentic.”

In [ ]:
from strands import Agent, tool

@tool
def ebike_lookup(model_name: str) -> dict:
    """
    Look up an e-bike model.

    Args:
        model_name: Exact product model name: CityLite, TrailPro, or CargoMax.

    Returns:
        Product details including price and range.
    """
    return get_ebike_price(model_name)

@tool
def arithmetic(operation: str, a: float, b: float) -> float:
    """
    Perform arithmetic.

    Args:
        operation: add, subtract, multiply, or divide
        a: first number
        b: second number
    """
    return calculator(operation, a, b)

In [ ]:
from strands import Agent, tool

@tool
def ebike_lookup(model_name: str) -> dict:
    """
    Look up an e-bike model.

    Args:
        model_name: Exact product model name: CityLite, TrailPro, or CargoMax.

    Returns:
        Product details including price and range.
    """
    return get_ebike_price(model_name)

@tool
def arithmetic(operation: str, a: float, b: float) -> float:
    """
    Perform arithmetic.

    Args:
        operation: add, subtract, multiply, or divide
        a: first number
        b: second number
    """
    return calculator(operation, a, b)

In [ ]:
system_prompt = """
You are an e-bike sales analysis agent.

Rules:
- Use ebike_lookup for product facts.
- Use arithmetic for calculations instead of mental arithmetic.
- Continue using tools until you have enough verified information.
- Keep the final answer concise and show the key calculation.
"""

agent = Agent(
    model=MODEL_ID,
    tools=[ebike_lookup, arithmetic],
    system_prompt=system_prompt,
)

In [ ]:
result = agent(
    "A customer wants 4 TrailPro bikes. "
    "What is the total before tax, and what range does each bike have?"
)

print("\nRESULT:\n", result)

### Compare the code

**Manual Bedrock loop**

```text
call model
inspect toolUse
dispatch Python function
build toolResult
call model again
repeat...
```

**Strands**

```python
agent = Agent(model=..., tools=[...], system_prompt=...)
agent("goal")
```

The loop still exists — the framework is handling it.

### Trainer question

> “What did Strands remove?”

Expected answer: **boilerplate orchestration**, not the concepts of model, tools, actions, observations, and iteration.

## Optional: show built-in Strands tools

The Strands tools package includes ready-made tools. This is useful to show the difference between:

- a custom Python tool you wrote
- a prebuilt tool supplied by a framework

In [ ]:
# Optional cell
# from strands_tools import calculator as strands_calculator, current_time
#
# utility_agent = Agent(
#     model=MODEL_ID,
#     tools=[strands_calculator, current_time],
#     system_prompt="Use tools whenever the question requires them."
# )
#
# utility_agent(
#     "What time is it, and what is 17.5 multiplied by 48? Use the available tools."
# )

# 5. Demo 4 — MCP with the AWS Documentation MCP Server

Until now, tools were imported directly into our Python process.

MCP changes the integration boundary:

```text
Strands Agent
     |
     | MCP client
     v
AWS Documentation MCP Server
     |
     +--> search_documentation
     +--> read_documentation
     +--> recommendations
     ...
```

The AWS Labs MCP repository provides an **AWS Documentation MCP Server** that exposes AWS documentation capabilities through the Model Context Protocol.



### MCP prerequisite: install `uv`

The AWS Documentation MCP Server is commonly launched with `uvx`.

Check whether it is available:

In [ ]:
import shutil, os

os.environ['PATH'] = r'C:\Users\salim\.local\bin' + os.pathsep + os.environ.get('PATH', '')
print("uv :", shutil.which("uv"))
print("uvx:", shutil.which("uvx"))

### Build a cross-platform MCP command

AWS's documentation server uses a slightly different launch form on Windows.

In [ ]:
import os
import platform

def aws_docs_mcp_server_params():
    from mcp import StdioServerParameters

    env = os.environ.copy()
    env.update({
        "FASTMCP_LOG_LEVEL": "ERROR",
        "AWS_DOCUMENTATION_PARTITION": "aws",
    })

    if platform.system() == "Windows":
        # AWS Labs documents the Windows form using `uv tool run --from ... <exe>`.
        return StdioServerParameters(
            command="uv",
            args=[
                "tool",
                "run",
                "--from",
                "awslabs.aws-documentation-mcp-server@latest",
                "awslabs.aws-documentation-mcp-server.exe",
            ],
            env=env,
        )

    return StdioServerParameters(
        command="uvx",
        args=["awslabs.aws-documentation-mcp-server@latest"],
        env=env,
    )

### Connect Strands to the MCP server

The MCP server owns the documentation tools. Strands discovers and uses them through `MCPClient`.

>  Keep the MCP client in a context manager so the server process is closed cleanly when the demo finishes.

In [ ]:
# Run this cell only when uv/uvx and internet access are available.

import os, platform
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client
from strands.tools.mcp import MCPClient

os.environ['PATH'] = r'C:\Users\salim\.local\bin' + os.pathsep + os.environ.get('PATH', '')

def get_mcp_params():
    env = os.environ.copy()
    env.update({'FASTMCP_LOG_LEVEL': 'ERROR', 'AWS_DOCUMENTATION_PARTITION': 'aws'})
    if platform.system() == 'Windows':
        return StdioServerParameters(
            command='uv',
            args=['tool', 'run', '--from', 'awslabs.aws-documentation-mcp-server@latest', 'awslabs.aws-documentation-mcp-server.exe'],
            env=env,
        )
    return StdioServerParameters(
        command='uvx',
        args=['awslabs.aws-documentation-mcp-server@latest'],
        env=env,
    )

_devnull = open(os.devnull, 'w')
try:
    mcp_client = MCPClient(lambda: stdio_client(get_mcp_params(), errlog=_devnull))
    with mcp_client:
        aws_doc_tools = mcp_client.list_tools_sync()
        print('Tools discovered from MCP server:')
        for t in aws_doc_tools:
            tool_name = getattr(t, 'tool_name', None) or getattr(t, 'name', None) or str(t)
            print('-', tool_name)
        docs_agent = Agent(
            model=MODEL_ID,
            tools=aws_doc_tools,
            system_prompt=(
                'You are an AWS documentation assistant. '
                'Use MCP documentation tools for AWS factual questions. '
                'Prefer documentation evidence over memory. '
                'Keep the response concise.'
            ),
        )
        result = docs_agent(
            'Search the AWS documentation for Amazon Bedrock Guardrails. '
            'Briefly explain what guardrails are used for.'
        )
        print('\nMCP AGENT RESULT:\n', result)
finally:
    _devnull.close()

— what changed?

Compare:

```text
BEFORE MCP

Agent process
 ├── Python tool A
 ├── Python tool B
 └── Python tool C
```

with:

```text
WITH MCP

Agent
  |
MCP Client
  |
MCP Server
  ├── Tool A
  ├── Tool B
  └── Tool C
```

### Key message

**MCP standardizes how an AI application connects to external tools/resources.**

The model still reasons about which capability it needs. The protocol changes **how that capability is exposed and accessed**.

# 6. A2A — Where agent-to-agent interoperability fits

Do not confuse **MCP** with **A2A**.

Use this picture:

```text
                 A2A
 Research Agent <-----> Analysis Agent
      |                      |
      | MCP                  | MCP
      v                      v
   Tools                  Tools
```

- **MCP:** agent/application ↔ tools and resources
- **A2A:** agent ↔ agent

AWS provides a larger interoperability sample:

https://github.com/aws-samples/sample-agent-interoperability

That sample demonstrates agents exposed through the A2A protocol and tools consumed through MCP. It includes Strands Agents, Amazon Bedrock AgentCore, and a multi-cloud setup.

### Why we do not deploy it live in this notebook

The full sample intentionally demonstrates production-style interoperability and has significantly more prerequisites than this 40-minute teaching module. Use its architecture as the advanced reference after students understand the simpler local loop.

## Optional  simulation — specialist delegation

This is **not an A2A protocol implementation**. It is only a quick conceptual bridge showing why one agent might delegate work to another specialist.

For a real A2A implementation, use the AWS interoperability repository above.

In [ ]:
# Conceptual only: an agent-as-a-tool pattern, not the A2A wire protocol.

pricing_specialist = Agent(
    model=MODEL_ID,
    tools=[ebike_lookup, arithmetic],
    system_prompt=(
        "You are a pricing specialist. "
        "Use tools for product facts and arithmetic."
    ),
)

@tool
def ask_pricing_specialist(question: str) -> str:
    """Delegate an e-bike pricing question to the pricing specialist."""
    return str(pricing_specialist(question))

supervisor = Agent(
    model=MODEL_ID,
    tools=[ask_pricing_specialist],
    system_prompt=(
        "You are a supervisor. Delegate pricing questions to the pricing specialist."
    ),
)

# Uncomment to run:
# supervisor("Find the cost of 2 CargoMax bikes.")

# 7. Cleanup

The core notebook intentionally creates **no persistent AWS resources**.

It uses:
- Bedrock Runtime API calls
- in-memory Python functions
- local Strands agent objects
- a temporary MCP subprocess inside a context manager

So there is no S3 bucket, DynamoDB table, Lambda function, AgentCore Runtime, CloudFormation stack, or other infrastructure to delete.

The MCP server process is closed when the `with mcp_client:` block exits.

In [ ]:
# Optional local cleanup
for variable_name in [
    "agent",
    "pricing_specialist",
    "supervisor",
    "bedrock",
    "session",
]:
    globals().pop(variable_name, None)

print("Local demo objects removed.")
print("No persistent AWS resources were created by the core notebook.")

# 8. Recap

End with these four statements:

### Function calling
**The model chooses a tool; the application executes it.**

### ReAct / agentic loop
**The system can repeat choose → act → observe until the goal is complete.**

### Strands
**A framework can manage the model-driven agent loop and reduce orchestration boilerplate.**

### MCP
**A standard way to expose and consume tools/resources.**

### A2A
**A standard way for independent agents to collaborate.**

---

## Suggested final question to students

Classify each scenario:

1. “The model asks my application to run a calculator function.” → **Function/tool calling**
2. “The system searches, calculates, checks another source, then answers.” → **Agentic loop / ReAct pattern**
3. “My Strands agent discovers documentation tools from a separate server.” → **MCP**
4. “A travel agent delegates work to an independently deployed finance agent.” → **A2A**